In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Semana12_Supervisado_G5").getOrCreate()
df_clusters = spark.read.parquet("/home/jovyan/work/datos_etiquetados_alojamientos")
df_supervisado = df_clusters.withColumnRenamed("prediction", "label")
print(f"Total: {df_supervisado.count()}")
df_supervisado.groupBy("label").count().orderBy("label").show()

In [ ]:
train_data, test_data = df_supervisado.randomSplit([0.7, 0.3], seed=42)
from pyspark.ml.feature import VectorAssembler, StandardScaler
feature_cols = ["precio_num","puntuacion_num","estrellas_num","ciudad_cat","zona_cat","tipo_cat","plataforma_cat"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
train_vec = assembler.transform(train_data)
test_vec = assembler.transform(test_data)
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")
scaler_model = scaler.fit(train_vec)
train_scaled = scaler_model.transform(train_vec)
test_scaled = scaler_model.transform(test_vec)

In [ ]:
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
dt = DecisionTreeClassifier(featuresCol="scaledFeatures", labelCol="label", maxDepth=5, seed=42)
dt_model = dt.fit(train_scaled)
dt_pred = dt_model.transform(test_scaled)
evaluator = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
acc_dt = evaluator.evaluate(dt_pred)
print(f"Árbol Decisión: {acc_dt*100:.2f}%")

In [ ]:
from pyspark.ml.classification import RandomForestClassifier
rf = RandomForestClassifier(featuresCol="scaledFeatures", labelCol="label", numTrees=20, seed=42)
rf_model = rf.fit(train_scaled)
rf_pred = rf_model.transform(test_scaled)
acc_rf = evaluator.evaluate(rf_pred)
print(f"Random Forest: {acc_rf*100:.2f}%")

In [ ]:
from pyspark.ml.classification import LinearSVC, OneVsRest
svm_bin = LinearSVC(featuresCol="scaledFeatures", labelCol="label", maxIter=10)
ovr_svm = OneVsRest(classifier=svm_bin, labelCol="label", featuresCol="scaledFeatures")
svm_model = ovr_svm.fit(train_scaled)
svm_pred = svm_model.transform(test_scaled)
acc_svm = evaluator.evaluate(svm_pred)
print(f"SVM: {acc_svm*100:.2f}%")


In [ ]:
from pyspark.ml.classification import LogisticRegression
lr = LogisticRegression(featuresCol="scaledFeatures", labelCol="label", family="multinomial", maxIter=10)
lr_model = lr.fit(train_scaled)
lr_pred = lr_model.transform(test_scaled)
acc_lr = evaluator.evaluate(lr_pred)
print(f"Regresión Logística: {acc_lr*100:.2f}%")

In [ ]:
print("="*50)
print("RESUMEN ACCURACY")
print(f"Árbol: {acc_dt*100:.2f}% | RF: {acc_rf*100:.2f}% | SVM: {acc_svm*100:.2f}% | LR: {acc_lr*100:.2f}%")
print("="*50)

In [ ]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler, StandardScaler

feature_cols_reg = ["puntuacion_num","estrellas_num","ciudad_cat","zona_cat","tipo_cat","plataforma_cat"]
assembler_reg = VectorAssembler(inputCols=feature_cols_reg, outputCol="features_reg")
df_reg = assembler_reg.transform(df_supervisado)
scaler_reg = StandardScaler(inputCol="features_reg", outputCol="scaledFeatures_reg")
df_reg_scaled = scaler_reg.fit(df_reg).transform(df_reg).withColumnRenamed("precio_num", "label_precio")
train_reg, test_reg = df_reg_scaled.randomSplit([0.7,0.3], seed=42)
lr_reg = LinearRegression(featuresCol="scaledFeatures_reg", labelCol="label_precio", maxIter=10)
lr_reg_model = lr_reg.fit(train_reg)
pred_reg = lr_reg_model.transform(test_reg)
evaluator_r2 = RegressionEvaluator(labelCol="label_precio", metricName="r2")
r2 = evaluator_r2.evaluate(pred_reg)
print(f"Regresión Lineal - R2: {r2*100:.2f}%")

In [ ]:
print("""
TICKET DE SALIDA - SEMANA 12
1. El problema son los datos, no los algoritmos. La clasificación tiene alta precisión porque los clusters de K-Means están bien definidos. La regresión lineal tiene R2 bajo porque faltan variables como ubicación exacta, temporada, servicios del hotel, capacidad, etc.

2. En el mundo real se necesitarían: latitud/longitud, distancia al centro/playa, temporada alta/baja, servicios (piscina, wifi, desayuno), capacidad de habitaciones, competencia en la zona, eventos especiales. Se obtendrían con scraping adicional y APIs de geolocalización.
""")